# Step 2 — Velocity visualization

This release notebook contains the updated implementation. A duplicated earlier implementation was removed to prevent the pipeline from running twice.


In [ ]:
#!/usr/bin/env python
# coding: utf-8

"""
RNA-velocity proxy visualization on UMAP and spatial coordinates.

Workflow
--------
1. Load delta matrix, current-expression matrix and metadata.
2. Align common cells and common genes.
3. Calculate initial delta magnitude, PCA and UMAP.
4. Use KMeans to divide the initial UMAP into two groups.
5. Retain the main group and remove the distant group.
6. Recalculate delta magnitude, PCA and UMAP after filtering.
7. Plot:
   - Grid-averaged UMAP velocity arrows
   - Spatial smoothed delta-gradient arrows
   - Gaussian-smoothed UMAP delta-gradient arrows
8. For all arrow plots, retain arrows whose lengths are between
   the 5th and 95th percentiles.
9. Save removed cells, retained cells, QC summaries and figures.
"""

# =====================================================
# Imports
# =====================================================

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from scipy.ndimage import gaussian_filter

import umap


warnings.filterwarnings("ignore", category=FutureWarning)


# =====================================================
# 1. File configuration
# =====================================================

NORM_MODE = os.environ.get("SPATIAL_VELO_NORM_MODE", "2D").upper()

_CWD_DATA_DIR = os.path.join(os.getcwd(), "data", "test_data")
_DEFAULT_WORKDIR = (
    _CWD_DATA_DIR
    if os.path.isdir(_CWD_DATA_DIR)
    else os.path.join("..", "data", "test_data")
)
BASE_DIR = os.path.abspath(
    os.environ.get("SPATIAL_VELO_WORKDIR", _DEFAULT_WORKDIR)
)

DELTA_FILE = os.path.join(
    BASE_DIR,
    "pipeline_out",
    f"mode_{NORM_MODE}",
    "out_part3",
    "delta_export_prob_median.csv"
)

# If this file exists, use it directly as the current matrix.
CURRENT_FILE = os.path.join(
    BASE_DIR,
    "transcripts_cytoplasm_matrix_common.csv"
)

# Used only when CURRENT_FILE does not exist.
TX_FILE = os.path.join(
    BASE_DIR,
    "xenium_test_tx_file.csv"
)

META_FILE = os.path.join(
    BASE_DIR,
    "xenium_test_metadata.csv"
)

OUT_DIR = os.path.join(
    BASE_DIR,
    "pipeline_out",
    f"mode_{NORM_MODE}",
    "out_part3"
)

os.makedirs(OUT_DIR, exist_ok=True)


# =====================================================
# 2. Transcript-column configuration
# =====================================================

TX_CELL_COL = "cell"
TX_GENE_COL = "Target"
TX_COMP_COL = "CellComp"
TX_X_COL = "global_x"
TX_Y_COL = "global_y"

# Fallback column names
TX_CELL_FALLBACK = "cell"
TX_GENE_FALLBACK = "target"
TX_X_FALLBACK = "x"
TX_Y_FALLBACK = "y"

CYTOPLASM_LABEL = "cytoplasm"


# =====================================================
# 3. Metadata-column configuration
# =====================================================

META_CELL_COL = "cell_id"
META_X_COL = "CenterX_global_px"
META_Y_COL = "CenterY_global_px"


# =====================================================
# 4. Analysis parameters
# =====================================================

# NORM_MODE is configured above

TOP_VAR_GENES_FOR_PCA = 1000
PCA_COMPONENTS = 30

# Number of expression-space neighbors used to calculate
# neighbor-adjusted delta magnitude.
N_NEIGHBORS = 30

# UMAP settings
UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.3
UMAP_RANDOM_STATE = 42


# =====================================================
# 5. Distant-group filtering settings
# =====================================================

# True: identify and remove the distant UMAP group.
# False: retain all cells.
REMOVE_DISTANT_UMAP_GROUP = True

# The current UMAP appears to contain two groups.
UMAP_GROUP_N_CLUSTERS = 2

# None means automatically retain the largest group.
#
# After inspecting umap_groups_before_filtering.png,
# this can be manually changed to 0 or 1.
KEEP_UMAP_CLUSTER = None

# Print a warning if more than this fraction is removed.
MAX_EXPECTED_REMOVED_FRACTION = 0.40


# =====================================================
# 6. Arrow filtering settings
# =====================================================

# Retain arrows between the 5th and 95th percentiles.
ARROW_LOWER_QUANTILE = 0.05
ARROW_UPPER_QUANTILE = 0.95


# =====================================================
# 7. UMAP grid-arrow settings
# =====================================================

UMAP_GRID_N = 25
UMAP_ARROW_SCALE = 0.15
UMAP_POINT_SIZE = 20
UMAP_CMAP_VMAX = 25


# =====================================================
# 8. Spatial gradient-arrow settings
# =====================================================

SPATIAL_GRID_N = 40
SPATIAL_SMOOTH_SIGMA = 1.5
SPATIAL_ARROW_SCALE = 0.035
SPATIAL_POINT_SIZE = 52
SPATIAL_ALPHA_POINTS = 0.35
SPATIAL_CMAP_VMAX = 25

# This is the matplotlib quiver display scale.
SPATIAL_QUIVER_SCALE = 0.0002


# =====================================================
# 9. Gaussian-smoothed UMAP settings
# =====================================================

UMAP_SMOOTH_GRID_N = 50
UMAP_SMOOTH_SIGMA = 1.0
UMAP_SMOOTH_ARROW_SCALE = 0.1
UMAP_SMOOTH_QUIVER_SCALE = 0.3


# =====================================================
# 10. Helper functions
# =====================================================

def resolve_column(
    df,
    primary,
    fallback=None,
    required=True
):
    """
    Return primary if it exists, otherwise return fallback.
    """

    if primary in df.columns:
        return primary

    if fallback is not None and fallback in df.columns:
        return fallback

    if required:
        raise KeyError(
            f"Cannot find column '{primary}'"
            + (
                f" or fallback '{fallback}'"
                if fallback is not None
                else ""
            )
            + f".\nAvailable columns:\n{list(df.columns)}"
        )

    return None


def standardize_transcript_table(tx):
    """
    Standardize transcript table columns to:
    cell, gene, CellComp, global_x, global_y.
    """

    cell_col = resolve_column(
        tx,
        TX_CELL_COL,
        TX_CELL_FALLBACK,
        required=True
    )

    gene_col = resolve_column(
        tx,
        TX_GENE_COL,
        TX_GENE_FALLBACK,
        required=True
    )

    x_col = resolve_column(
        tx,
        TX_X_COL,
        TX_X_FALLBACK,
        required=True
    )

    y_col = resolve_column(
        tx,
        TX_Y_COL,
        TX_Y_FALLBACK,
        required=True
    )

    comp_col = resolve_column(
        tx,
        TX_COMP_COL,
        fallback=None,
        required=False
    )

    out = pd.DataFrame()

    out["cell"] = tx[cell_col].astype(str)
    out["gene"] = tx[gene_col].astype(str)

    out["global_x"] = pd.to_numeric(
        tx[x_col],
        errors="coerce"
    )

    out["global_y"] = pd.to_numeric(
        tx[y_col],
        errors="coerce"
    )

    if comp_col is not None:
        out["CellComp"] = tx[comp_col].astype(str)
    else:
        out["CellComp"] = np.nan

    out = out.dropna(
        subset=[
            "cell",
            "gene",
            "global_x",
            "global_y"
        ]
    )

    return out


def build_current_matrix_from_transcripts(tx_std):
    """
    Build a cell-by-gene count matrix from cytoplasmic transcripts.
    """

    if "CellComp" not in tx_std.columns:
        raise RuntimeError(
            "Transcript table does not contain CellComp."
        )

    comp = (
        tx_std["CellComp"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    cyto_mask = comp == CYTOPLASM_LABEL.lower()

    if not cyto_mask.any():
        raise RuntimeError(
            f"No transcripts matched '{CYTOPLASM_LABEL}'. "
            "Check TX_COMP_COL and CYTOPLASM_LABEL."
        )

    tx_cyto = tx_std.loc[
        cyto_mask,
        ["cell", "gene"]
    ].copy()

    current_df = (
        tx_cyto
        .groupby(["cell", "gene"])
        .size()
        .unstack(fill_value=0)
    )

    current_df.index = current_df.index.astype(str)
    current_df.columns = current_df.columns.astype(str)

    return current_df


def find_current_matrix():
    """
    Load CURRENT_FILE if it exists.
    Otherwise build the current matrix from TX_FILE.
    """

    if os.path.exists(CURRENT_FILE):

        print(f"Using current matrix:\n{CURRENT_FILE}")

        current_df = pd.read_csv(
            CURRENT_FILE,
            index_col=0
        )

        current_df.index = current_df.index.astype(str)
        current_df.columns = current_df.columns.astype(str)

        return current_df

    print(
        "CURRENT_FILE was not found.\n"
        f"Building current matrix from:\n{TX_FILE}"
    )

    if not os.path.exists(TX_FILE):
        raise FileNotFoundError(
            "Neither CURRENT_FILE nor TX_FILE exists.\n\n"
            f"CURRENT_FILE:\n{CURRENT_FILE}\n\n"
            f"TX_FILE:\n{TX_FILE}"
        )

    tx_raw = pd.read_csv(TX_FILE)

    tx_std = standardize_transcript_table(tx_raw)

    standardized_tx_file = os.path.join(
        OUT_DIR,
        "transcripts_standardized.csv"
    )

    tx_std.to_csv(
        standardized_tx_file,
        index=False
    )

    print(
        f"Saved standardized transcripts:\n"
        f"{standardized_tx_file}"
    )

    current_df = build_current_matrix_from_transcripts(
        tx_std
    )

    current_output_file = os.path.join(
        OUT_DIR,
        "current_matrix_from_transcripts.csv"
    )

    current_df.to_csv(current_output_file)

    print(
        f"Saved current matrix:\n"
        f"{current_output_file}"
    )

    return current_df


def compute_delta_magnitude(
    current_df,
    delta_df,
    n_neighbors
):
    """
    Calculate neighbor-adjusted delta magnitude.

    Nearest neighbors are identified in the current-expression
    matrix. For each cell:

        diff = cell delta vector - mean current vector of neighbors

    The L2 norm of diff is used as delta magnitude.
    """

    n_cells = current_df.shape[0]

    if n_cells < 2:
        raise RuntimeError(
            "At least two cells are required."
        )

    current_values = current_df.to_numpy(
        dtype=np.float32
    )

    delta_values = delta_df.to_numpy(
        dtype=np.float32
    )

    # Query one additional neighbor because the cell itself
    # is usually returned as the nearest neighbor.
    n_query_neighbors = min(
        n_neighbors + 1,
        n_cells
    )

    print(
        f"Finding approximately "
        f"{n_query_neighbors - 1} neighbors per cell..."
    )

    nbrs = NearestNeighbors(
        n_neighbors=n_query_neighbors,
        n_jobs=-1,
        algorithm="auto"
    )

    nbrs.fit(current_values)

    _, neighbor_indices = nbrs.kneighbors(
        current_values
    )

    delta_magnitude = np.zeros(
        n_cells,
        dtype=np.float32
    )

    for i in tqdm(
        range(n_cells),
        desc="Computing delta magnitude",
        unit="cell"
    ):

        neighbors = neighbor_indices[i]

        # Remove the cell itself.
        neighbors = neighbors[
            neighbors != i
        ]

        if len(neighbors) > n_neighbors:
            neighbors = neighbors[:n_neighbors]

        if len(neighbors) == 0:
            neighbor_mean = current_values[i]
        else:
            neighbor_mean = current_values[
                neighbors
            ].mean(axis=0)

        diff = (
            delta_values[i]
            - neighbor_mean
        )

        delta_magnitude[i] = np.linalg.norm(
            diff
        )

    magnitude_series = pd.Series(
        delta_magnitude,
        index=delta_df.index,
        name="delta_magnitude"
    )

    return delta_magnitude, magnitude_series


def compute_pca_and_velocity(delta_df):
    """
    Select top-variable delta genes, standardize them,
    calculate PCA coordinates and project delta vectors
    into PCA space.
    """

    print("Selecting top-variable delta genes...")

    gene_variance = (
        delta_df
        .var(axis=0)
        .sort_values(ascending=False)
    )

    n_top_genes = min(
        TOP_VAR_GENES_FOR_PCA,
        len(gene_variance)
    )

    top_genes = (
        gene_variance
        .head(n_top_genes)
        .index
        .tolist()
    )

    delta_pca_input = (
        delta_df[top_genes]
        .fillna(0.0)
        .astype(np.float32)
    )

    print(
        f"Using {len(top_genes):,} genes for PCA."
    )

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        delta_pca_input.to_numpy()
    )

    n_components_eff = min(
        PCA_COMPONENTS,
        X_scaled.shape[1],
        X_scaled.shape[0] - 1
    )

    if n_components_eff < 2:
        raise RuntimeError(
            "At least two PCA components are required."
        )

    pca = PCA(
        n_components=n_components_eff,
        random_state=0
    )

    X_pca = pca.fit_transform(X_scaled)

    # A vector should be divided by the scaler standard
    # deviations, but should not be mean-centered.
    scale_values = scaler.scale_.copy()
    scale_values[scale_values == 0] = 1.0

    delta_velocity_scaled = (
        delta_pca_input.to_numpy()
        / scale_values
    )

    velocity_pca = (
        delta_velocity_scaled
        @ pca.components_.T
    )

    explained_variance = (
        pca.explained_variance_ratio_.sum()
    )

    print(
        f"PCA completed: {X_pca.shape}"
    )

    print(
        f"Explained variance: "
        f"{explained_variance:.2%}"
    )

    return {
        "top_genes": top_genes,
        "scaler": scaler,
        "pca": pca,
        "X_scaled": X_scaled,
        "X_pca": X_pca,
        "velocity_pca": velocity_pca
    }


def compute_umap_embedding(X_pca):
    """
    Calculate a two-dimensional UMAP embedding.
    """

    n_cells = X_pca.shape[0]

    if n_cells < 3:
        raise RuntimeError(
            "At least three cells are required for UMAP."
        )

    n_neighbors_eff = min(
        UMAP_N_NEIGHBORS,
        n_cells - 1
    )

    print(
        "Computing UMAP with "
        f"n_neighbors={n_neighbors_eff}, "
        f"min_dist={UMAP_MIN_DIST}..."
    )

    reducer = umap.UMAP(
        n_neighbors=n_neighbors_eff,
        min_dist=UMAP_MIN_DIST,
        n_components=2,
        random_state=UMAP_RANDOM_STATE
    )

    X_umap = reducer.fit_transform(X_pca)

    return reducer, X_umap


def assign_grid_bins(
    x_values,
    y_values,
    x_edges,
    y_edges
):
    """
    Assign each point to an x/y grid bin.

    Values at the maximum boundaries are assigned
    to the final bins.
    """

    x_bin = np.digitize(
        x_values,
        x_edges
    ) - 1

    y_bin = np.digitize(
        y_values,
        y_edges
    ) - 1

    x_bin = np.clip(
        x_bin,
        0,
        len(x_edges) - 2
    )

    y_bin = np.clip(
        y_bin,
        0,
        len(y_edges) - 2
    )

    return x_bin, y_bin


def filter_arrows_by_quantile(
    x,
    y,
    u,
    v,
    valid_mask=None,
    lower_quantile=0.05,
    upper_quantile=0.95,
    plot_name="arrows"
):
    """
    Retain arrows whose lengths are between the specified
    lower and upper quantiles.

    The default retains the 5th-95th percentiles.
    """

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    u = np.asarray(
        u,
        dtype=float
    )

    v = np.asarray(
        v,
        dtype=float
    )

    if not (
        x.shape == y.shape == u.shape == v.shape
    ):
        raise ValueError(
            "x, y, u and v must have identical shapes."
        )

    finite_mask = (
        np.isfinite(x)
        & np.isfinite(y)
        & np.isfinite(u)
        & np.isfinite(v)
    )

    if valid_mask is None:
        valid_mask = finite_mask
    else:
        valid_mask = (
            np.asarray(
                valid_mask,
                dtype=bool
            )
            & finite_mask
        )

    arrow_lengths = np.sqrt(
        u ** 2
        + v ** 2
    )

    valid_lengths = arrow_lengths[
        valid_mask
    ]

    if len(valid_lengths) == 0:

        print(
            f"Warning: no valid arrows for {plot_name}."
        )

        empty = np.full_like(
            arrow_lengths,
            np.nan,
            dtype=float
        )

        return (
            empty,
            empty,
            empty,
            empty,
            np.nan,
            np.nan,
            0,
            0
        )

    lower_cutoff = np.quantile(
        valid_lengths,
        lower_quantile
    )

    upper_cutoff = np.quantile(
        valid_lengths,
        upper_quantile
    )

    keep_mask = (
        valid_mask
        & (arrow_lengths >= lower_cutoff)
        & (arrow_lengths <= upper_cutoff)
    )

    x_filtered = np.where(
        keep_mask,
        x,
        np.nan
    )

    y_filtered = np.where(
        keep_mask,
        y,
        np.nan
    )

    u_filtered = np.where(
        keep_mask,
        u,
        np.nan
    )

    v_filtered = np.where(
        keep_mask,
        v,
        np.nan
    )

    n_valid = int(valid_mask.sum())
    n_retained = int(keep_mask.sum())

    print(
        f"\n{plot_name} arrow filtering:"
    )

    print(
        f"  Quantile range: "
        f"{lower_quantile:.0%}-{upper_quantile:.0%}"
    )

    print(
        f"  Length cutoff: "
        f"{lower_cutoff:.6g} to {upper_cutoff:.6g}"
    )

    print(
        f"  Retained arrows: "
        f"{n_retained:,}/{n_valid:,}"
    )

    return (
        x_filtered,
        y_filtered,
        u_filtered,
        v_filtered,
        lower_cutoff,
        upper_cutoff,
        n_valid,
        n_retained
    )


# This table records the 5%-95% cutoff values
# used for each arrow plot.
arrow_filter_records = []


# =====================================================
# 11. Load data
# =====================================================

print("=" * 72)
print("Loading data")
print("=" * 72)

print(f"NORM_MODE = {NORM_MODE}")

if NORM_MODE.upper() not in {
    "2D",
    "3D"
}:
    raise ValueError(
        f"NORM_MODE must be '2D' or '3D', "
        f"not '{NORM_MODE}'."
    )

if not os.path.exists(DELTA_FILE):
    raise FileNotFoundError(
        f"DELTA_FILE does not exist:\n{DELTA_FILE}"
    )

if not os.path.exists(META_FILE):
    raise FileNotFoundError(
        f"META_FILE does not exist:\n{META_FILE}"
    )

print(f"Loading delta matrix:\n{DELTA_FILE}")

delta = pd.read_csv(
    DELTA_FILE,
    index_col=0
)

delta.index = delta.index.astype(str)
delta.columns = delta.columns.astype(str)

current = find_current_matrix()

print(f"Loading metadata:\n{META_FILE}")

meta = pd.read_csv(META_FILE)


# =====================================================
# 12. Validate metadata
# =====================================================

required_meta_columns = [
    META_CELL_COL,
    META_X_COL,
    META_Y_COL
]

for column in required_meta_columns:

    if column not in meta.columns:

        raise KeyError(
            f"Metadata column '{column}' was not found.\n"
            f"Available columns:\n{list(meta.columns)}"
        )

meta["cell"] = meta[
    META_CELL_COL
].astype(str)

if meta["cell"].duplicated().any():

    duplicate_count = int(
        meta["cell"]
        .duplicated()
        .sum()
    )

    print(
        f"Warning: metadata contains "
        f"{duplicate_count:,} duplicated cell rows."
    )

    print(
        "Only the first metadata row for each cell "
        "will be retained."
    )

    meta = (
        meta
        .drop_duplicates(
            subset="cell",
            keep="first"
        )
        .reset_index(drop=True)
    )


# =====================================================
# 13. Align cells and genes
# =====================================================

common_cells = (
    delta.index
    .intersection(current.index)
    .intersection(meta["cell"])
)

common_genes = (
    delta.columns
    .intersection(current.columns)
)

if len(common_cells) == 0:
    raise RuntimeError(
        "No common cells were found among delta, "
        "current and metadata."
    )

if len(common_genes) == 0:
    raise RuntimeError(
        "No common genes were found between delta "
        "and current."
    )

delta = delta.loc[
    common_cells,
    common_genes
].copy()

current = current.loc[
    common_cells,
    common_genes
].copy()

meta = (
    meta[
        meta["cell"].isin(common_cells)
    ]
    .copy()
    .reset_index(drop=True)
)

# Convert matrices to numeric values.
delta = (
    delta
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .fillna(0.0)
)

current = (
    current
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .fillna(0.0)
)

print(
    f"Common cells: {len(common_cells):,}"
)

print(
    f"Common genes: {len(common_genes):,}"
)

print(
    f"Delta shape: {delta.shape}"
)

print(
    f"Current shape: {current.shape}"
)

if NORM_MODE.upper() == "3D":

    print(
        "Warning: NORM_MODE=3D is selected, but this "
        "plotting script does not calculate geometric "
        "3D nuclear-distance normalization."
    )


# =====================================================
# 14. Initial analysis before distant-group removal
# =====================================================

print("\n" + "=" * 72)
print("Initial analysis before distant-group filtering")
print("=" * 72)

initial_delta_adj, initial_delta_adj_series = (
    compute_delta_magnitude(
        current_df=current,
        delta_df=delta,
        n_neighbors=N_NEIGHBORS
    )
)

initial_delta_file = os.path.join(
    OUT_DIR,
    "delta_magnitude_before_umap_filter.csv"
)

initial_delta_adj_series.to_csv(
    initial_delta_file
)

print(
    f"Saved initial delta magnitude:\n"
    f"{initial_delta_file}"
)

initial_pca_results = compute_pca_and_velocity(
    delta
)

initial_reducer, initial_X_umap = (
    compute_umap_embedding(
        initial_pca_results["X_pca"]
    )
)


# =====================================================
# 15. Identify and remove distant UMAP group
# =====================================================

if REMOVE_DISTANT_UMAP_GROUP:

    print("\n" + "=" * 72)
    print("Detecting distant UMAP group")
    print("=" * 72)

    kmeans = KMeans(
        n_clusters=UMAP_GROUP_N_CLUSTERS,
        random_state=UMAP_RANDOM_STATE,
        n_init=20
    )

    umap_group_labels = kmeans.fit_predict(
        initial_X_umap
    )

    group_counts = (
        pd.Series(
            umap_group_labels,
            name="group"
        )
        .value_counts()
        .sort_values(ascending=False)
    )

    print("Detected UMAP group sizes:")

    for group_label, group_size in group_counts.items():

        print(
            f"  Group {group_label}: "
            f"{group_size:,} cells"
        )

    if KEEP_UMAP_CLUSTER is None:

        # Automatically retain the largest group.
        keep_group = int(
            group_counts.index[0]
        )

    else:

        keep_group = int(
            KEEP_UMAP_CLUSTER
        )

        if keep_group not in group_counts.index:

            raise ValueError(
                f"KEEP_UMAP_CLUSTER={keep_group} "
                "does not exist.\n"
                f"Available groups: "
                f"{list(group_counts.index)}"
            )

    keep_mask = (
        umap_group_labels == keep_group
    )

    remove_mask = ~keep_mask

    retained_cells = delta.index[
        keep_mask
    ]

    removed_cells = delta.index[
        remove_mask
    ]

    removed_fraction = (
        len(removed_cells)
        / len(delta)
    )

    print(
        f"\nRetained group: {keep_group}"
    )

    print(
        f"Retained cells: "
        f"{len(retained_cells):,}"
    )

    print(
        f"Removed cells: "
        f"{len(removed_cells):,}"
    )

    print(
        f"Removed fraction: "
        f"{removed_fraction:.2%}"
    )

    if (
        removed_fraction
        > MAX_EXPECTED_REMOVED_FRACTION
    ):

        print(
            "\nWARNING: the removed group contains more than "
            f"{MAX_EXPECTED_REMOVED_FRACTION:.0%} of cells."
        )

        print(
            "Inspect umap_groups_before_filtering.png and "
            "set KEEP_UMAP_CLUSTER manually if necessary."
        )

    cluster_centers = kmeans.cluster_centers_

    if UMAP_GROUP_N_CLUSTERS == 2:

        center_distance = np.linalg.norm(
            cluster_centers[0]
            - cluster_centers[1]
        )

        print(
            f"Distance between UMAP group centers: "
            f"{center_distance:.4f}"
        )


    # -------------------------------------------------
    # Cell-level audit table
    # -------------------------------------------------

    audit_df = pd.DataFrame({
        "cell": delta.index.astype(str),

        "UMAP1_before_filter":
            initial_X_umap[:, 0],

        "UMAP2_before_filter":
            initial_X_umap[:, 1],

        "umap_group":
            umap_group_labels,

        "retained":
            keep_mask,

        "delta_magnitude_before_filter":
            initial_delta_adj,

        "total_current_transcripts":
            current.sum(axis=1).to_numpy(),

        "detected_current_genes":
            (current > 0).sum(axis=1).to_numpy()
    })

    audit_df = audit_df.merge(
        meta,
        on="cell",
        how="left"
    )

    audit_file = os.path.join(
        OUT_DIR,
        "umap_group_audit.csv"
    )

    audit_df.to_csv(
        audit_file,
        index=False
    )


    # -------------------------------------------------
    # Save removed cells
    # -------------------------------------------------

    removed_file = os.path.join(
        OUT_DIR,
        "removed_distant_umap_cells.csv"
    )

    audit_df.loc[
        ~audit_df["retained"]
    ].to_csv(
        removed_file,
        index=False
    )


    # -------------------------------------------------
    # Save retained cells
    # -------------------------------------------------

    retained_file = os.path.join(
        OUT_DIR,
        "retained_main_umap_cells.csv"
    )

    audit_df.loc[
        audit_df["retained"]
    ].to_csv(
        retained_file,
        index=False
    )


    # -------------------------------------------------
    # Group QC summary
    # -------------------------------------------------

    group_summary = (
        audit_df
        .groupby("umap_group")
        .agg(
            n_cells=(
                "cell",
                "size"
            ),

            median_delta_magnitude=(
                "delta_magnitude_before_filter",
                "median"
            ),

            mean_delta_magnitude=(
                "delta_magnitude_before_filter",
                "mean"
            ),

            median_total_transcripts=(
                "total_current_transcripts",
                "median"
            ),

            mean_total_transcripts=(
                "total_current_transcripts",
                "mean"
            ),

            median_detected_genes=(
                "detected_current_genes",
                "median"
            ),

            mean_detected_genes=(
                "detected_current_genes",
                "mean"
            )
        )
        .sort_values(
            "n_cells",
            ascending=False
        )
    )

    summary_file = os.path.join(
        OUT_DIR,
        "umap_group_qc_summary.csv"
    )

    group_summary.to_csv(
        summary_file
    )

    print("\nUMAP group QC summary:")
    print(group_summary)


    # -------------------------------------------------
    # Plot original groups
    # -------------------------------------------------

    plt.figure(
        figsize=(9, 7)
    )

    group_scatter = plt.scatter(
        initial_X_umap[:, 0],
        initial_X_umap[:, 1],
        c=umap_group_labels,
        cmap="tab10",
        s=8,
        alpha=0.75,
        edgecolors="none"
    )

    plt.colorbar(
        group_scatter,
        label="KMeans UMAP group"
    )

    plt.title(
        "UMAP groups before filtering\n"
        f"Retained group = {keep_group}"
    )

    plt.xlabel("UMAP1")
    plt.ylabel("UMAP2")
    plt.tight_layout()

    group_plot_file = os.path.join(
        OUT_DIR,
        "umap_groups_before_filtering.png"
    )

    plt.savefig(
        group_plot_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


    # -------------------------------------------------
    # Plot retained and removed cells
    # -------------------------------------------------

    plt.figure(
        figsize=(9, 7)
    )

    plt.scatter(
        initial_X_umap[
            keep_mask,
            0
        ],
        initial_X_umap[
            keep_mask,
            1
        ],
        s=8,
        alpha=0.65,
        edgecolors="none",
        label=(
            f"Retained: "
            f"{int(keep_mask.sum()):,}"
        )
    )

    plt.scatter(
        initial_X_umap[
            remove_mask,
            0
        ],
        initial_X_umap[
            remove_mask,
            1
        ],
        s=12,
        alpha=0.85,
        edgecolors="none",
        label=(
            f"Removed: "
            f"{int(remove_mask.sum()):,}"
        )
    )

    plt.title(
        "Retained and removed UMAP cells"
    )

    plt.xlabel("UMAP1")
    plt.ylabel("UMAP2")
    plt.legend()
    plt.tight_layout()

    retained_removed_plot = os.path.join(
        OUT_DIR,
        "umap_retained_vs_removed.png"
    )

    plt.savefig(
        retained_removed_plot,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(f"\nSaved group audit:\n{audit_file}")
    print(f"Saved removed cells:\n{removed_file}")
    print(f"Saved retained cells:\n{retained_file}")
    print(f"Saved QC summary:\n{summary_file}")
    print(f"Saved group plot:\n{group_plot_file}")
    print(
        f"Saved retained/removed plot:\n"
        f"{retained_removed_plot}"
    )


    # -------------------------------------------------
    # Apply the cell mask
    # -------------------------------------------------

    delta = delta.loc[
        retained_cells
    ].copy()

    current = current.loc[
        retained_cells
    ].copy()

    meta = (
        meta[
            meta["cell"].isin(retained_cells)
        ]
        .copy()
        .reset_index(drop=True)
    )


    # -------------------------------------------------
    # Recalculate all quantities after filtering
    # -------------------------------------------------

    print("\n" + "=" * 72)
    print("Recalculating after distant-group removal")
    print("=" * 72)

    delta_adj, delta_adj_series = (
        compute_delta_magnitude(
            current_df=current,
            delta_df=delta,
            n_neighbors=N_NEIGHBORS
        )
    )

    pca_results = compute_pca_and_velocity(
        delta
    )

    X_pca = pca_results[
        "X_pca"
    ]

    velocity_pca = pca_results[
        "velocity_pca"
    ]

    reducer, X_umap = (
        compute_umap_embedding(
            X_pca
        )
    )

else:

    print(
        "\nREMOVE_DISTANT_UMAP_GROUP=False. "
        "All cells will be retained."
    )

    delta_adj = initial_delta_adj

    delta_adj_series = (
        initial_delta_adj_series
    )

    pca_results = initial_pca_results

    X_pca = pca_results[
        "X_pca"
    ]

    velocity_pca = pca_results[
        "velocity_pca"
    ]

    reducer = initial_reducer
    X_umap = initial_X_umap


# =====================================================
# 16. Save final delta magnitude and final UMAP
# =====================================================

final_delta_file = os.path.join(
    OUT_DIR,
    "delta_magnitude_per_cell.csv"
)

delta_adj_series.to_csv(
    final_delta_file
)

print(
    f"\nSaved final delta magnitude:\n"
    f"{final_delta_file}"
)

umap_df = pd.DataFrame(
    X_umap,
    columns=[
        "UMAP1",
        "UMAP2"
    ],
    index=delta.index.astype(str)
)

umap_df["delta_magnitude"] = (
    delta_adj_series
    .loc[umap_df.index]
    .to_numpy()
)

umap_df_reset = (
    umap_df
    .reset_index()
)

umap_df_reset = umap_df_reset.rename(
    columns={
        umap_df_reset.columns[0]:
            "cell"
    }
)

umap_df_reset["cell"] = (
    umap_df_reset["cell"]
    .astype(str)
)

umap_with_meta = pd.merge(
    umap_df_reset,
    meta,
    on="cell",
    how="left"
)

umap_meta_output = os.path.join(
    OUT_DIR,
    "umap_with_meta.csv"
)

umap_with_meta.to_csv(
    umap_meta_output,
    index=False
)

print(
    f"Saved final UMAP coordinates:\n"
    f"{umap_meta_output}"
)


# =====================================================
# 17. Grid-averaged UMAP velocity arrows
# =====================================================

print("\n" + "=" * 72)
print("Building grid-averaged UMAP velocity arrows")
print("=" * 72)

if velocity_pca.shape[1] < 2:
    raise RuntimeError(
        "velocity_pca contains fewer than two dimensions."
    )

vel2d = velocity_pca[:, :2]

x_min = X_umap[:, 0].min()
x_max = X_umap[:, 0].max()

y_min = X_umap[:, 1].min()
y_max = X_umap[:, 1].max()

x_edges = np.linspace(
    x_min,
    x_max,
    UMAP_GRID_N + 1
)

y_edges = np.linspace(
    y_min,
    y_max,
    UMAP_GRID_N + 1
)

x_bin, y_bin = assign_grid_bins(
    X_umap[:, 0],
    X_umap[:, 1],
    x_edges,
    y_edges
)

X_centers = []
Y_centers = []
U_grid = []
V_grid = []

for i in range(UMAP_GRID_N):

    for j in range(UMAP_GRID_N):

        grid_mask = (
            (x_bin == i)
            & (y_bin == j)
        )

        if grid_mask.sum() == 0:
            continue

        x_center = X_umap[
            grid_mask,
            0
        ].mean()

        y_center = X_umap[
            grid_mask,
            1
        ].mean()

        u_value = vel2d[
            grid_mask,
            0
        ].mean()

        v_value = vel2d[
            grid_mask,
            1
        ].mean()

        speed = np.sqrt(
            u_value ** 2
            + v_value ** 2
        )

        if speed > 0:

            u_scaled = (
                u_value
                / speed
                * np.sqrt(speed)
                * UMAP_ARROW_SCALE
            )

            v_scaled = (
                v_value
                / speed
                * np.sqrt(speed)
                * UMAP_ARROW_SCALE
            )

        else:

            u_scaled = 0.0
            v_scaled = 0.0

        X_centers.append(x_center)
        Y_centers.append(y_center)
        U_grid.append(u_scaled)
        V_grid.append(v_scaled)


X_centers = np.asarray(
    X_centers,
    dtype=float
)

Y_centers = np.asarray(
    Y_centers,
    dtype=float
)

U_grid = np.asarray(
    U_grid,
    dtype=float
)

V_grid = np.asarray(
    V_grid,
    dtype=float
)


(
    X_centers_filtered,
    Y_centers_filtered,
    U_grid_filtered,
    V_grid_filtered,
    umap_grid_lower_cutoff,
    umap_grid_upper_cutoff,
    umap_grid_valid_n,
    umap_grid_retained_n
) = filter_arrows_by_quantile(
    x=X_centers,
    y=Y_centers,
    u=U_grid,
    v=V_grid,
    valid_mask=None,
    lower_quantile=ARROW_LOWER_QUANTILE,
    upper_quantile=ARROW_UPPER_QUANTILE,
    plot_name="UMAP grid"
)

arrow_filter_records.append({
    "plot": "umap_velocity_grid",
    "lower_quantile": ARROW_LOWER_QUANTILE,
    "upper_quantile": ARROW_UPPER_QUANTILE,
    "lower_length_cutoff": umap_grid_lower_cutoff,
    "upper_length_cutoff": umap_grid_upper_cutoff,
    "valid_arrows": umap_grid_valid_n,
    "retained_arrows": umap_grid_retained_n
})


print("Plotting grid-averaged UMAP velocity figure...")

plt.figure(
    figsize=(8, 6)
)

scatter = plt.scatter(
    umap_df["UMAP1"],
    umap_df["UMAP2"],
    c=umap_df["delta_magnitude"],
    cmap="coolwarm",
    s=UMAP_POINT_SIZE,
    alpha=0.60,
    edgecolors="none",
    vmax=UMAP_CMAP_VMAX
)

plt.colorbar(
    scatter,
    label="delta_magnitude"
)

plt.quiver(
    X_centers_filtered,
    Y_centers_filtered,
    U_grid_filtered,
    V_grid_filtered,
    angles="xy",
    scale_units="xy",
    scale=1,
    color="black",
    alpha=0.8,
    width=0.003
)

if REMOVE_DISTANT_UMAP_GROUP:

    plt.title(
        "UMAP with grid-averaged velocity arrows\n"
        "Distant group removed; arrow length: 5%-95%"
    )

else:

    plt.title(
        "UMAP with grid-averaged velocity arrows\n"
        "Arrow length: 5%-95%"
    )

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.tight_layout()

umap_velocity_output = os.path.join(
    OUT_DIR,
    "umap_velocity_grid.png"
)

plt.savefig(
    umap_velocity_output,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print(
    f"Saved UMAP velocity plot:\n"
    f"{umap_velocity_output}"
)


# =====================================================
# 18. Spatial smoothed delta-gradient arrows
# =====================================================

print("\n" + "=" * 72)
print("Building spatial delta-gradient arrows")
print("=" * 72)

meta_indexed = (
    meta
    .set_index("cell")
    .loc[delta.index]
)

X_real = pd.to_numeric(
    meta_indexed[META_X_COL],
    errors="coerce"
).to_numpy()

Y_real = pd.to_numeric(
    meta_indexed[META_Y_COL],
    errors="coerce"
).to_numpy()

delta_mag = (
    delta_adj_series
    .loc[meta_indexed.index]
    .to_numpy()
)

if (
    np.isnan(X_real).any()
    or np.isnan(Y_real).any()
):

    raise ValueError(
        "NaN values were found in spatial coordinates: "
        f"{META_X_COL}, {META_Y_COL}"
    )

x_edges = np.linspace(
    X_real.min(),
    X_real.max(),
    SPATIAL_GRID_N + 1
)

y_edges = np.linspace(
    Y_real.min(),
    Y_real.max(),
    SPATIAL_GRID_N + 1
)

x_centers = 0.5 * (
    x_edges[:-1]
    + x_edges[1:]
)

y_centers = 0.5 * (
    y_edges[:-1]
    + y_edges[1:]
)

X_grid, Y_grid = np.meshgrid(
    x_centers,
    y_centers
)

delta_grid = np.zeros_like(
    X_grid,
    dtype=float
)

count_grid = np.zeros_like(
    X_grid,
    dtype=float
)

x_bin, y_bin = assign_grid_bins(
    X_real,
    Y_real,
    x_edges,
    y_edges
)

for i in range(SPATIAL_GRID_N):

    for j in range(SPATIAL_GRID_N):

        grid_mask = (
            (x_bin == i)
            & (y_bin == j)
        )

        if grid_mask.sum() > 0:

            delta_grid[j, i] = (
                delta_mag[
                    grid_mask
                ].mean()
            )

            count_grid[j, i] = (
                grid_mask.sum()
            )


delta_smooth = gaussian_filter(
    delta_grid,
    sigma=SPATIAL_SMOOTH_SIGMA
)

grad_y, grad_x = np.gradient(
    delta_smooth
)

gradient_magnitude = np.sqrt(
    grad_x ** 2
    + grad_y ** 2
)

gradient_scaling = np.sqrt(
    gradient_magnitude
)

U_spatial = (
    grad_x
    / (gradient_magnitude + 1e-8)
    * gradient_scaling
    * SPATIAL_ARROW_SCALE
)

V_spatial = (
    grad_y
    / (gradient_magnitude + 1e-8)
    * gradient_scaling
    * SPATIAL_ARROW_SCALE
)

spatial_valid_mask = (
    count_grid > 0
)

U_spatial_masked = np.where(
    spatial_valid_mask,
    U_spatial,
    0.0
)

V_spatial_masked = np.where(
    spatial_valid_mask,
    V_spatial,
    0.0
)


(
    X_spatial_filtered,
    Y_spatial_filtered,
    U_spatial_filtered,
    V_spatial_filtered,
    spatial_lower_cutoff,
    spatial_upper_cutoff,
    spatial_valid_n,
    spatial_retained_n
) = filter_arrows_by_quantile(
    x=X_grid,
    y=Y_grid,
    u=U_spatial_masked,
    v=V_spatial_masked,
    valid_mask=spatial_valid_mask,
    lower_quantile=ARROW_LOWER_QUANTILE,
    upper_quantile=ARROW_UPPER_QUANTILE,
    plot_name="Spatial gradient"
)

arrow_filter_records.append({
    "plot": "spatial_delta_gradient",
    "lower_quantile": ARROW_LOWER_QUANTILE,
    "upper_quantile": ARROW_UPPER_QUANTILE,
    "lower_length_cutoff": spatial_lower_cutoff,
    "upper_length_cutoff": spatial_upper_cutoff,
    "valid_arrows": spatial_valid_n,
    "retained_arrows": spatial_retained_n
})


print("Plotting spatial delta-gradient figure...")

plt.figure(
    figsize=(12, 12)
)

scatter = plt.scatter(
    X_real,
    Y_real,
    c=delta_mag,
    cmap="coolwarm",
    s=SPATIAL_POINT_SIZE,
    edgecolors="none",
    alpha=SPATIAL_ALPHA_POINTS,
    vmax=SPATIAL_CMAP_VMAX
)

plt.colorbar(
    scatter,
    label="delta_magnitude"
)

plt.quiver(
    X_spatial_filtered,
    Y_spatial_filtered,
    U_spatial_filtered,
    V_spatial_filtered,
    angles="xy",
    scale_units="xy",
    scale=SPATIAL_QUIVER_SCALE,
    color="black",
    alpha=0.9,
    width=0.0015
)

plt.xlabel(META_X_COL)
plt.ylabel(META_Y_COL)

plt.title(
    "Spatial cells with smoothed delta-gradient arrows\n"
    "Arrow length: 5%-95%"
)

plt.axis("equal")
plt.tight_layout()

spatial_output = os.path.join(
    OUT_DIR,
    "spatial_delta_gradient.png"
)

plt.savefig(
    spatial_output,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print(
    f"Saved spatial gradient plot:\n"
    f"{spatial_output}"
)


# =====================================================
# 19. Gaussian-smoothed UMAP arrows
# =====================================================

print("\n" + "=" * 72)
print("Building Gaussian-smoothed UMAP arrows")
print("=" * 72)

x_min = X_umap[:, 0].min()
x_max = X_umap[:, 0].max()

y_min = X_umap[:, 1].min()
y_max = X_umap[:, 1].max()

x_edges = np.linspace(
    x_min,
    x_max,
    UMAP_SMOOTH_GRID_N + 1
)

y_edges = np.linspace(
    y_min,
    y_max,
    UMAP_SMOOTH_GRID_N + 1
)

x_centers = 0.5 * (
    x_edges[:-1]
    + x_edges[1:]
)

y_centers = 0.5 * (
    y_edges[:-1]
    + y_edges[1:]
)

X_grid, Y_grid = np.meshgrid(
    x_centers,
    y_centers
)

delta_grid = np.zeros_like(
    X_grid,
    dtype=float
)

count_grid = np.zeros_like(
    X_grid,
    dtype=float
)

x_bin, y_bin = assign_grid_bins(
    X_umap[:, 0],
    X_umap[:, 1],
    x_edges,
    y_edges
)

for i in range(UMAP_SMOOTH_GRID_N):

    for j in range(UMAP_SMOOTH_GRID_N):

        grid_mask = (
            (x_bin == i)
            & (y_bin == j)
        )

        if grid_mask.sum() > 0:

            delta_grid[j, i] = (
                delta_adj[
                    grid_mask
                ].mean()
            )

            count_grid[j, i] = (
                grid_mask.sum()
            )


delta_smooth = gaussian_filter(
    delta_grid,
    sigma=UMAP_SMOOTH_SIGMA
)

grad_y, grad_x = np.gradient(
    delta_smooth
)

gradient_magnitude = np.sqrt(
    grad_x ** 2
    + grad_y ** 2
)

gradient_scaling = np.sqrt(
    gradient_magnitude
)

U_umap_smooth = (
    grad_x
    / (gradient_magnitude + 1e-8)
    * gradient_scaling
    * UMAP_SMOOTH_ARROW_SCALE
)

V_umap_smooth = (
    grad_y
    / (gradient_magnitude + 1e-8)
    * gradient_scaling
    * UMAP_SMOOTH_ARROW_SCALE
)

umap_smooth_valid_mask = (
    count_grid > 0
)

U_umap_smooth_masked = np.where(
    umap_smooth_valid_mask,
    U_umap_smooth,
    0.0
)

V_umap_smooth_masked = np.where(
    umap_smooth_valid_mask,
    V_umap_smooth,
    0.0
)


(
    X_umap_smooth_filtered,
    Y_umap_smooth_filtered,
    U_umap_smooth_filtered,
    V_umap_smooth_filtered,
    umap_smooth_lower_cutoff,
    umap_smooth_upper_cutoff,
    umap_smooth_valid_n,
    umap_smooth_retained_n
) = filter_arrows_by_quantile(
    x=X_grid,
    y=Y_grid,
    u=U_umap_smooth_masked,
    v=V_umap_smooth_masked,
    valid_mask=umap_smooth_valid_mask,
    lower_quantile=ARROW_LOWER_QUANTILE,
    upper_quantile=ARROW_UPPER_QUANTILE,
    plot_name="Gaussian-smoothed UMAP"
)

arrow_filter_records.append({
    "plot": "umap_gaussian_smoothed",
    "lower_quantile": ARROW_LOWER_QUANTILE,
    "upper_quantile": ARROW_UPPER_QUANTILE,
    "lower_length_cutoff": umap_smooth_lower_cutoff,
    "upper_length_cutoff": umap_smooth_upper_cutoff,
    "valid_arrows": umap_smooth_valid_n,
    "retained_arrows": umap_smooth_retained_n
})


print("Plotting Gaussian-smoothed UMAP figure...")

plt.figure(
    figsize=(8, 6)
)

scatter = plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=delta_adj,
    cmap="coolwarm",
    s=UMAP_POINT_SIZE,
    alpha=0.55,
    edgecolors="none",
    vmax=UMAP_CMAP_VMAX
)

plt.colorbar(
    scatter,
    label="delta_magnitude"
)

plt.quiver(
    X_umap_smooth_filtered,
    Y_umap_smooth_filtered,
    U_umap_smooth_filtered,
    V_umap_smooth_filtered,
    angles="xy",
    scale_units="xy",
    scale=UMAP_SMOOTH_QUIVER_SCALE,
    color="black",
    alpha=0.8,
    width=0.002
)

plt.title(
    "UMAP with Gaussian-smoothed delta-gradient arrows\n"
    "Arrow length: 5%-95%"
)

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.tight_layout()

umap_smooth_output = os.path.join(
    OUT_DIR,
    "umap_gaussian_smoothed.png"
)

plt.savefig(
    umap_smooth_output,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print(
    f"Saved Gaussian-smoothed UMAP:\n"
    f"{umap_smooth_output}"
)


# =====================================================
# 20. Save arrow-filter summary
# =====================================================

arrow_filter_summary = pd.DataFrame(
    arrow_filter_records
)

arrow_filter_summary_file = os.path.join(
    OUT_DIR,
    "arrow_filter_5_95_summary.csv"
)

arrow_filter_summary.to_csv(
    arrow_filter_summary_file,
    index=False
)

print(
    f"\nSaved arrow-filter summary:\n"
    f"{arrow_filter_summary_file}"
)

print("\nArrow-filter summary:")
print(arrow_filter_summary)


# =====================================================
# 21. Final summary
# =====================================================

print("\n" + "=" * 72)
print("Analysis completed")
print("=" * 72)

print(
    f"Final number of cells: "
    f"{len(delta):,}"
)

print(
    f"Final number of genes: "
    f"{delta.shape[1]:,}"
)

print(
    f"Output directory:\n{OUT_DIR}"
)

print("\nMain output files:")

if REMOVE_DISTANT_UMAP_GROUP:

    print(
        "  1. umap_groups_before_filtering.png"
    )

    print(
        "  2. umap_retained_vs_removed.png"
    )

    print(
        "  3. removed_distant_umap_cells.csv"
    )

    print(
        "  4. retained_main_umap_cells.csv"
    )

    print(
        "  5. umap_group_qc_summary.csv"
    )

print(
    "  6. delta_magnitude_per_cell.csv"
)

print(
    "  7. umap_with_meta.csv"
)

print(
    "  8. umap_velocity_grid.png"
)

print(
    "  9. spatial_delta_gradient.png"
)

print(
    " 10. umap_gaussian_smoothed.png"
)

print(
    " 11. arrow_filter_5_95_summary.csv"
)

print("\nDone.")